# Inspect Consolidated AET Datasets

Load and visualize the two AET source datasets for January 2000.

Sources (see `catalog/variables.yml` → `aet`):
- MOD16A2 v061 (`ET_500m`, kg m-2 per 8-day composite) — local consolidated per-year NC
- SSEBop (`actual_et`, mm/month) — remote zarr at the USGS GDP STAC OSN endpoint

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import xarray as xr

DATASTORE = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/nhf-datastore")
PROJECT = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets")

TARGET_TIME = "2000-01-15"
TARGET_YEAR = 2000

## Load both datasets

In [ ]:
datasets = {
    "MOD16A2 v061 (ET_500m)": {
        "path": DATASTORE / "mod16a2_v061" / f"mod16a2_v061_{TARGET_YEAR}_consolidated.nc",
        "var": "ET_500m",
        "units": "kg m-2 / 8-day",
    },
    "SSEBop (actual_et)": {
        "zarr_url": "s3://mdmf/gdp/ssebopeta_monthly.zarr/",
        "storage_options": {
            "anon": True,
            "endpoint_url": "https://usgs.osn.mghpcc.org/",
        },
        "var": "actual_et",
        "units": "mm/month",
    },
}


def _open(info):
    if "zarr_url" in info:
        return xr.open_zarr(
            info["zarr_url"],
            storage_options=info.get("storage_options", {}),
            consolidated=True,
        )
    return xr.open_dataset(info["path"])


opened = {}
for label, info in datasets.items():
    try:
        if "path" in info and not info["path"].exists():
            print(f"SKIP {label}: {info['path']} not found (run fetch first)")
            continue
        ds = _open(info)
    except Exception as exc:
        print(f"SKIP {label}: open failed: {exc}")
        continue
    opened[label] = (ds, info)
    print(f"{label}: {list(ds.data_vars)} | time: {ds.time.values[0]} .. {ds.time.values[-1]} | shape: {dict(ds.sizes)}")

## Dataset representations

In [ ]:
for label, (ds, _) in opened.items():
    print(f"{'=' * 60}\n{label}\n{'=' * 60}")
    display(ds)

## Plot January 2000

In [ ]:
n = len(opened)
if n == 0:
    print("No datasets available yet. Run the fetch commands first.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    axes = axes.flatten() if n > 1 else [axes]

    for idx, (label, (ds, info)) in enumerate(opened.items()):
        ax = axes[idx]
        var = info["var"]
        da = ds[var].sel(time=TARGET_TIME, method="nearest")
        actual_time = str(da.time.values)[:10]

        da.plot(ax=ax, cmap="YlOrRd", robust=True)
        ax.set_title(f"{label}\n{actual_time} | {info['units']}", fontsize=11)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

    for idx in range(len(opened), len(axes)):
        axes[idx].set_visible(False)

    fig.suptitle(f"AET — nearest to {TARGET_TIME}", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

## Clean up

In [ ]:
for label, (ds, _) in opened.items():
    ds.close()